# Proyecto Final

In [1]:
import pandas as pd
import os

### Objetivo 1: Predicción del Retorno del Día Siguiente
- Objetivo: entrenar un modelo de machine learning para predecir si una acción aumentará más de un 1% en el siguiente día de trading, utilizando información de los últimos 30 días de datos del mercado del S&P 500.
- Descripción: el modelo aprovechará indicadores técnicos, acción del precio, dinámica del volumen, medidas de volatilidad y variables rezagadas para aprender patrones de mercado a corto plazo que preceden movimientos positivos significativos del precio. Esta tarea se plantea como un problema de clasificación binaria, donde la variable objetivo indica si el retorno del día siguiente supera el umbral del 1%.
- Propósito: evaluar la viabilidad de la predicción de retornos a corto plazo utilizando datos históricos de mercado, evitando estrictamente el look-ahead bias y preservando la estructura temporal de las series financieras.

### Objetivo 2: Detección de Crisis / Riesgo de Mercado
- Objetivo: entrenar un modelo para detectar patrones de alerta temprana que suelen ocurrir antes de caídas importantes del mercado o periodos de volatilidad extrema en el S&P 500.
- Descripción: este modelo se centra en identificar comportamientos anómalos en la volatilidad, el volumen y la dinámica del precio que puedan indicar un aumento del riesgo de mercado o el inicio de una crisis. La tarea puede abordarse mediante detección de anomalías o clasificación de riesgo, aprendiendo qué se considera un comportamiento “normal” del mercado y señalando desviaciones respecto a él.
- Propósito: proporcionar una capa de conciencia de riesgo que complemente el modelo de predicción de retornos, destacando condiciones de mercado inestables en las que la confianza predictiva y las estrategias de trading deberían ajustarse.

### Origen de los datos y descripción
Los datos de este proyecto provienen de Kaggle, del dataset Advanced Stock Dataset creado por baidalinadilzhan, que a su vez se basa en datos descargados de Yahoo Finance API, cubriendo aproximadamente los últimos cinco años de mercado para los componentes del índice S&P 500. Este conjunto de datos incluye observaciones diarias ajustadas por eventos corporativos como dividendos y splits, y está diseñado específicamente para análisis financiero y modelado de series temporales.
En total, el dataset contiene más de 600 000 registros diarios con 73 características que incluyen precios (Open, High, Low, Close), volumen, indicadores técnicos como medias móviles y RSI, métricas de volatilidad y múltiples variables con información histórica rezagada (lags), lo que lo hace adecuado para tareas de predicción de retornos, clasificación direccional y análisis de riesgo en machine learning financiero.

## Cargar datos y crear DataFrame

In [ ]:
# Ruta absoluta al archivo .parquet que contiene el dataset de entrenamiento
file_path = r"......"

In [ ]:
# Se lee el archivo parquet usando pandas
df = pd.read_parquet(file_path, engine="pyarrow")

## Análisis descriptivo

In [ ]:
# Mostrar las primeras filas del dataset
df.head()

In [ ]:
# Ver el número de filas y columnas
df.shape

In [ ]:
# Listar todas las columnas del DataFrame
df.columns.tolist()

In [ ]:
# Información general del DataFrame
df.info()

In [ ]:
# Contar valores nulos por columna
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df.describe()

## Columnas del Dataset
- DATE: Fecha del día de trading.
- TICKER: Símbolo bursátil de la acción.
- OPEN: Precio de apertura del día.
- HIGH: Precio máximo alcanzado durante el día.
- LOW: Precio mínimo alcanzado durante el día.
- CLOSE: Precio de cierre del día.
- VOLUME: Número total de acciones negociadas.
- DIVIDENDS: Dividendos pagados ese día (si existen).
- STOCK_SPLITS: Split de acciones ocurrido ese día (si existe).
- SMA_5: Media móvil simple del cierre en los últimos 5 días.
- SMA_10: Media móvil simple del cierre en los últimos 10 días.
- SMA_20: Media móvil simple del cierre en los últimos 20 días.
- SMA_50: Media móvil simple del cierre en los últimos 50 días.
- EMA_12: Media móvil exponencial de 12 días.
- EMA_26: Media móvil exponencial de 26 días.
- MACD: Diferencia entre EMA_12 y EMA_26 (indicador de momentum).
- MACD_SIGNAL: Media móvil del MACD.
- MACD_HISTOGRAM: Diferencia entre MACD y su señal.
- RSI: Índice de Fuerza Relativa (14 periodos).
- VOLATILITY: Volatilidad histórica del precio.
- BB_MIDDLE: Banda central de Bollinger (media móvil).
- BB_UPPER: Banda superior de Bollinger.
- BB_LOWER: Banda inferior de Bollinger.
- BB_WIDTH: Ancho de las bandas de Bollinger (medida de volatilidad relativa).
- BB_POSITION: Posición del precio dentro de las bandas de Bollinger.
- PRICE_CHANGE: Cambio diario del precio de cierre.
- PRICE_CHANGE_5D: Cambio acumulado del precio en los últimos 5 días.
- HIGH_LOW_RATIO: Relación entre precio máximo y mínimo.
- OPEN_CLOSE_RATIO: Relación entre apertura y cierre.
- VOLUME_SMA: Media móvil del volumen.
- VOLUME_RATIO: Volumen actual comparado con su media.
- CLOSE_LAG_1: Precio de cierre de hace 1 día.
- CLOSE_LAG_2: Precio de cierre de hace 2 días.
- CLOSE_LAG_3: Precio de cierre de hace 3 días.
- CLOSE_LAG_4: Precio de cierre de hace 4 días.
- CLOSE_LAG_5: Precio de cierre de hace 5 días.
- CLOSE_LAG_6: Precio de cierre de hace 6 días.
- CLOSE_LAG_7: Precio de cierre de hace 7 días.
- CLOSE_LAG_8: Precio de cierre de hace 8 días.
- CLOSE_LAG_9: Precio de cierre de hace 9 días.
- CLOSE_LAG_10: Precio de cierre de hace 10 días.
- VOLUME_LAG_1: Volumen de hace 1 día.
- VOLUME_LAG_2: Volumen de hace 2 días.
- VOLUME_LAG_3: Volumen de hace 3 días.
- VOLUME_LAG_4: Volumen de hace 4 días.
- VOLUME_LAG_5: Volumen de hace 5 días.
- VOLUME_LAG_6: Volumen de hace 6 días.
- VOLUME_LAG_7: Volumen de hace 7 días.
- VOLUME_LAG_8: Volumen de hace 8 días.
- VOLUME_LAG_9: Volumen de hace 9 días.
- VOLUME_LAG_10: Volumen de hace 10 días.
- PRICE_CHANGE_LAG_1: Cambio de precio de hace 1 día.
- PRICE_CHANGE_LAG_2: Cambio de precio de hace 2 días.
- PRICE_CHANGE_LAG_3: Cambio de precio de hace 3 días.
- PRICE_CHANGE_LAG_4: Cambio de precio de hace 4 días.
- PRICE_CHANGE_LAG_5: Cambio de precio de hace 5 días.
- PRICE_CHANGE_LAG_6: Cambio de precio de hace 6 días.
- PRICE_CHANGE_LAG_7: Cambio de precio de hace 7 días.
- PRICE_CHANGE_LAG_8: Cambio de precio de hace 8 días.
- PRICE_CHANGE_LAG_9: Cambio de precio de hace 9 días.
- PRICE_CHANGE_LAG_10: Cambio de precio de hace 10 días.
- RSI_LAG_1: RSI de hace 1 día.
- RSI_LAG_2: RSI de hace 2 días.
- RSI_LAG_3: RSI de hace 3 días.
- RSI_LAG_4: RSI de hace 4 días.
- RSI_LAG_5: RSI de hace 5 días.
- MACD_LAG_1: MACD de hace 1 día.
- MACD_LAG_2: MACD de hace 2 días.
- MACD_LAG_3: MACD de hace 3 días.
- MACD_LAG_4: MACD de hace 4 días.
- MACD_LAG_5: MACD de hace 5 días.
- VOLATILITY_LAG_1: Volatilidad de hace 1 día.
- VOLATILITY_LAG_2: Volatilidad de hace 2 días.